In [0]:
# ============================================================================
# GETYOURGUIDE PRICING FEATURES AUDIT - DATABRICKS VERSION
# ============================================================================

import pandas as pd
import numpy as np
from decimal import Decimal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set_palette("husl")

# ============================================================================
# 1. LOAD DATA
# ============================================================================

df = spark.sql("SELECT * FROM production.supply_analytics.pricing_features_audit_export").toPandas()

# ============================================================================
# 2. CONVERT DECIMAL TYPES
# ============================================================================

for col in df.columns:
    if df[col].dtype == 'object':
        try:
            if len(df[col]) > 0 and isinstance(df[col].iloc[0], Decimal):
                df[col] = df[col].astype(float)
        except:
            pass

numeric_cols = ['gmv_l12mo', 'avg_booking_value_l12mo', 'avg_tickets_per_booking_l12mo',
                'repeat_customer_rate_l12mo', 'gmv_percentile',
                'est_scale_pricing_uplift_eur', 'est_addon_uplift_eur']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ============================================================================
# 3. FILTER DATA
# ============================================================================

df_analysis = df[
    (df['uses_external_pricing'] == 0) &
    (df['tour_value_tier'].isin(['top_tier', 'high_value', 'medium_value']))
].copy()

print(f"📊 Analyzing {len(df_analysis):,} tours")
print(f"💰 Total GMV: €{df_analysis['gmv_l12mo'].sum() / 1e6:.1f}M")

# ============================================================================
# 4. FEATURE IMPACT ANALYSIS
# ============================================================================

features_to_analyze = {
    'Scale Pricing': 'has_any_scale_pricing',
    'Addons': 'has_addons_configured',
    'Group Pricing': 'has_group_pricing',
    'Capacity Limits': 'has_capacity_limits',
    'Cutoff Configuration': 'has_cutoff_configured'
}

impact_results = []

for feature_name, feature_col in features_to_analyze.items():
    with_feature = df_analysis[df_analysis[feature_col] == 1]
    without_feature = df_analysis[df_analysis[feature_col] == 0]
    
    aov_with = with_feature['avg_booking_value_l12mo'].mean()
    aov_without = without_feature['avg_booking_value_l12mo'].mean()
    
    impact_results.append({
        'Feature': feature_name,
        'Tours_With': len(with_feature),
        'Tours_Without': len(without_feature),
        'Adoption_Rate_%': round(100 * len(with_feature) / len(df_analysis), 1),
        'AOV_With': round(aov_with, 2),
        'AOV_Without': round(aov_without, 2),
        'AOV_Lift_EUR': round(aov_with - aov_without, 2),
        'AOV_Lift_%': round(100 * (aov_with / aov_without - 1), 1) if aov_without > 0 else 0,
        'GMV_With_M': round(with_feature['gmv_l12mo'].sum() / 1e6, 2),
        'GMV_Without_M': round(without_feature['gmv_l12mo'].sum() / 1e6, 2)
    })

impact_df = pd.DataFrame(impact_results)

print("\n" + "="*80)
print("FEATURE IMPACT SUMMARY")
print("="*80)
display(impact_df)

# ============================================================================
# 5. CHART 1: AOV COMPARISON
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 6))
valid_impact = impact_df[~impact_df['AOV_Lift_%'].isna()].copy()

if len(valid_impact) > 0:
    x = np.arange(len(valid_impact))
    width = 0.35
    
    ax.bar(x - width/2, valid_impact['AOV_With'], width, label='With Feature', color='#2ecc71')
    ax.bar(x + width/2, valid_impact['AOV_Without'], width, label='Without Feature', color='#e74c3c')
    
    ax.set_xlabel('Feature', fontsize=13, fontweight='bold')
    ax.set_ylabel('Average Order Value (EUR)', fontsize=13, fontweight='bold')
    ax.set_title('Average Order Value: With vs Without Features', fontsize=15, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(valid_impact['Feature'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 6. MARKET ANALYSIS
# ============================================================================

market_analysis = df_analysis.groupby('location_name').agg({
    'tour_id': 'count',
    'supplier_id': 'nunique',
    'gmv_l12mo': 'sum',
    'bookings_l12mo': 'sum',
    'has_any_scale_pricing': 'mean',
    'has_addons_configured': 'mean',
    'has_group_pricing': 'mean',
    'avg_booking_value_l12mo': 'mean',
    'est_scale_pricing_uplift_eur': 'sum',
    'est_addon_uplift_eur': 'sum'
}).reset_index()

market_analysis.columns = [
    'Location', 'Total_Tours', 'Total_Suppliers', 'GMV_L12M', 'Bookings_L12M',
    'Scale_Adoption', 'Addon_Adoption', 'Group_Adoption',
    'Avg_AOV', 'Scale_Opportunity', 'Addon_Opportunity'
]

market_analysis['GMV_M'] = (market_analysis['GMV_L12M'] / 1e6).round(2)
market_analysis['Scale_Adoption_%'] = (market_analysis['Scale_Adoption'] * 100).round(1)
market_analysis['Addon_Adoption_%'] = (market_analysis['Addon_Adoption'] * 100).round(1)
market_analysis['Scale_Opp_M'] = (market_analysis['Scale_Opportunity'] / 1e6).round(2)

market_analysis = market_analysis[market_analysis['Total_Tours'] >= 20].sort_values('GMV_M', ascending=False)

print("\n" + "="*80)
print("TOP 15 MARKETS")
print("="*80)
display(market_analysis[['Location', 'Total_Tours', 'GMV_M', 'Scale_Adoption_%', 'Addon_Adoption_%', 'Scale_Opp_M']].head(15))

# ============================================================================
# 7. CHART 2: MARKET ADOPTION
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 8))
top_markets = market_analysis.head(15)
x = np.arange(len(top_markets))
width = 0.25

ax.bar(x - width, top_markets['Scale_Adoption_%'], width, label='Scale Pricing', color='#3498db')
ax.bar(x, top_markets['Addon_Adoption_%'], width, label='Addons', color='#9b59b6')
ax.bar(x + width, top_markets['Group_Adoption'] * 100, width, label='Group Pricing', color='#f39c12')

ax.set_xlabel('Location', fontsize=13, fontweight='bold')
ax.set_ylabel('Adoption Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Feature Adoption Rates by Top Markets', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(top_markets['Location'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 8. ACTIVITY TYPE ANALYSIS
# ============================================================================

activity_analysis = df_analysis.groupby('activity_type').agg({
    'tour_id': 'count',
    'gmv_l12mo': 'sum',
    'has_any_scale_pricing': 'mean',
    'has_addons_configured': 'mean',
    'has_group_pricing': 'mean',
    'avg_booking_value_l12mo': 'mean',
    'feature_count': 'mean',
    'est_scale_pricing_uplift_eur': 'sum'
}).reset_index()

activity_analysis.columns = [
    'Activity_Type', 'Total_Tours', 'GMV_L12M', 'Scale_Adoption',
    'Addon_Adoption', 'Group_Adoption', 'Avg_AOV', 'Avg_Features', 'Scale_Opportunity'
]

activity_analysis['GMV_M'] = (activity_analysis['GMV_L12M'] / 1e6).round(2)
activity_analysis['Scale_Adoption_%'] = (activity_analysis['Scale_Adoption'] * 100).round(1)
activity_analysis = activity_analysis.sort_values('GMV_M', ascending=False)

print("\n" + "="*80)
print("ACTIVITY TYPE ANALYSIS")
print("="*80)
display(activity_analysis[['Activity_Type', 'Total_Tours', 'GMV_M', 'Scale_Adoption_%', 'Avg_AOV']].head(10))

# ============================================================================
# 9. CHART 3: ACTIVITY HEATMAP
# ============================================================================

fig, ax = plt.subplots(figsize=(13, 8))
activity_top = activity_analysis.head(10)
heatmap_data = activity_top[['Scale_Adoption', 'Addon_Adoption', 'Group_Adoption']].values.T * 100

im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(np.arange(len(activity_top)))
ax.set_yticks(np.arange(3))
ax.set_xticklabels(activity_top['Activity_Type'], rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(['Scale Pricing', 'Addons', 'Group Pricing'], fontsize=12)

for i in range(3):
    for j in range(len(activity_top)):
        ax.text(j, i, f'{heatmap_data[i, j]:.0f}%', ha="center", va="center", color="black", fontsize=10, fontweight='bold')

ax.set_title('Feature Adoption by Activity Type (%)', fontsize=15, fontweight='bold')
fig.colorbar(im, ax=ax, label='Adoption Rate (%)')

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 10. SUPPLIER SEGMENTATION
# ============================================================================

supplier_analysis = df_analysis.groupby('supplier_id').agg({
    'location_name': 'first',
    'tour_id': 'count',
    'gmv_l12mo': 'sum',
    'has_any_scale_pricing': 'mean',
    'feature_count': 'mean',
    'est_scale_pricing_uplift_eur': 'sum',
    'est_addon_uplift_eur': 'sum'
}).reset_index()

supplier_analysis.columns = [
    'Supplier_ID', 'Primary_Location', 'Num_Tours', 'Total_GMV',
    'Scale_Adoption', 'Avg_Features', 'Scale_Opportunity', 'Addon_Opportunity'
]

supplier_analysis['Total_Opportunity'] = supplier_analysis['Scale_Opportunity'] + supplier_analysis['Addon_Opportunity']

def segment_supplier(row):
    if row['Total_GMV'] >= 100000 and row['Avg_Features'] < 2:
        return 'VIP - Low Adoption'
    elif row['Total_GMV'] >= 50000 and row['Scale_Adoption'] < 0.5:
        return 'High Value - Scale Opportunity'
    elif row['Num_Tours'] >= 10 and row['Avg_Features'] < 1.5:
        return 'Portfolio - Enable Features'
    else:
        return 'Standard'

supplier_analysis['Segment'] = supplier_analysis.apply(segment_supplier, axis=1)
supplier_analysis = supplier_analysis[supplier_analysis['Num_Tours'] >= 3].sort_values('Total_GMV', ascending=False)

print("\n" + "="*80)
print("SUPPLIER SEGMENTATION")
print("="*80)
print(f"Total Suppliers: {len(supplier_analysis):,}\n")
display(supplier_analysis['Segment'].value_counts())
display(supplier_analysis[['Supplier_ID', 'Primary_Location', 'Num_Tours', 'Total_GMV', 'Avg_Features', 'Segment']].head(20))

# ============================================================================
# 11. CHART 4: SUPPLIER SEGMENTS
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 7))
segment_counts = supplier_analysis['Segment'].value_counts()
colors_pie = ['#e74c3c', '#3498db', '#f39c12', '#95a5a6']

wedges, texts, autotexts = ax.pie(segment_counts.values, labels=segment_counts.index, 
                                    autopct='%1.1f%%', colors=colors_pie[:len(segment_counts)], startangle=90,
                                    textprops={'fontsize': 12, 'fontweight': 'bold'})

ax.set_title('Supplier Segmentation Distribution', fontsize=15, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 12. FEATURE COMBINATION ANALYSIS
# ============================================================================

feature_combo = df_analysis.groupby('feature_count').agg({
    'tour_id': 'count',
    'gmv_l12mo': 'sum',
    'avg_booking_value_l12mo': 'mean',
    'avg_tickets_per_booking_l12mo': 'mean',
    'bookings_l12mo': 'mean'
}).reset_index()

feature_combo.columns = ['Feature_Count', 'Num_Tours', 'Total_GMV', 'Avg_AOV', 'Avg_Group_Size', 'Avg_Bookings']
feature_combo['GMV_M'] = (feature_combo['Total_GMV'] / 1e6).round(2)
feature_combo = feature_combo.sort_values('Feature_Count')

print("\n" + "="*80)
print("FEATURE COMBINATION ANALYSIS")
print("="*80)
display(feature_combo)

# ============================================================================
# 13. CHART 5: FEATURE COUNT VS AOV
# ============================================================================

fig, ax = plt.subplots(figsize=(11, 7))

ax.plot(feature_combo['Feature_Count'], feature_combo['Avg_AOV'], 
        marker='o', linewidth=3, markersize=12, color='#e74c3c', label='Avg AOV')

for i, row in feature_combo.iterrows():
    ax.text(row['Feature_Count'], row['Avg_AOV'] + 15, 
            f"€{row['Avg_AOV']:.0f}", ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Number of Features Enabled', fontsize=13, fontweight='bold')
ax.set_ylabel('Average Order Value (EUR)', fontsize=13, fontweight='bold')
ax.set_title('Tour Performance by Number of Features', fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 14. OPPORTUNITIES
# ============================================================================

scale_opps = df_analysis[df_analysis['scale_pricing_opportunity'] == 1].sort_values('gmv_l12mo', ascending=False)
addon_opps = df_analysis[df_analysis['addon_opportunity'] == 1].sort_values('gmv_l12mo', ascending=False)

total_scale_opp = scale_opps['est_scale_pricing_uplift_eur'].sum() / 1e6
total_addon_opp = addon_opps['est_addon_uplift_eur'].sum() / 1e6
total_opp = total_scale_opp + total_addon_opp

print("\n" + "="*80)
print("OPPORTUNITY SUMMARY")
print("="*80)
print(f"Scale Pricing Opportunities: {len(scale_opps):,} tours → €{total_scale_opp:.2f}M")
print(f"Addon Opportunities: {len(addon_opps):,} tours → €{total_addon_opp:.2f}M")
print(f"TOTAL OPPORTUNITY: €{total_opp:.2f}M\n")

display(scale_opps[['tour_id', 'tour_title', 'location_name', 'gmv_l12mo', 'bookings_l12mo', 'avg_booking_value_l12mo', 'est_scale_pricing_uplift_eur']].head(30))

# ============================================================================
# 15. CHART 6: OPPORTUNITY WATERFALL
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 7))

categories = ['Scale Pricing\nOpportunity', 'Addon\nOpportunity', 'Total\nOpportunity']
values = [total_scale_opp, total_addon_opp, total_opp]
colors = ['#3498db', '#9b59b6', '#2ecc71']

bars = ax.bar(categories, values, color=colors, alpha=0.85, edgecolor='black', linewidth=2)

for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'€{val:.2f}M', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Estimated Revenue Opportunity (EUR Millions)', fontsize=13, fontweight='bold')
ax.set_title('Revenue Opportunity Breakdown', fontsize=15, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 16. EXECUTIVE SUMMARY DASHBOARD
# ============================================================================

fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# Key metrics
ax1 = fig.add_subplot(gs[0, 0])
ax1.text(0.5, 0.7, f"{len(df_analysis):,}", ha='center', va='center', fontsize=40, fontweight='bold', color='#2c3e50')
ax1.text(0.5, 0.3, 'Total Tours Analyzed', ha='center', va='center', fontsize=14, fontweight='bold')
ax1.axis('off')
ax1.set_facecolor('#ecf0f1')

ax2 = fig.add_subplot(gs[0, 1])
ax2.text(0.5, 0.7, f"€{df_analysis['gmv_l12mo'].sum() / 1e6:.1f}M", ha='center', va='center', fontsize=40, fontweight='bold', color='#27ae60')
ax2.text(0.5, 0.3, 'Total GMV L12M', ha='center', va='center', fontsize=14, fontweight='bold')
ax2.axis('off')
ax2.set_facecolor('#ecf0f1')

ax3 = fig.add_subplot(gs[0, 2])
ax3.text(0.5, 0.7, f"€{total_opp:.2f}M", ha='center', va='center', fontsize=40, fontweight='bold', color='#e74c3c')
ax3.text(0.5, 0.3, 'Revenue Opportunity', ha='center', va='center', fontsize=14, fontweight='bold')
ax3.axis('off')
ax3.set_facecolor('#ecf0f1')

# Feature adoption
ax4 = fig.add_subplot(gs[1, :])
features_adoption = impact_df.sort_values('Adoption_Rate_%')
ax4.barh(features_adoption['Feature'], features_adoption['Adoption_Rate_%'], color='#3498db')
ax4.set_xlabel('Adoption Rate (%)', fontsize=13, fontweight='bold')
ax4.set_title('Current Feature Adoption Rates', fontsize=15, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)

for i, v in enumerate(features_adoption['Adoption_Rate_%']):
    ax4.text(v + 2, i, f'{v:.1f}%', va='center', fontweight='bold')

# AOV Lift
ax5 = fig.add_subplot(gs[2, :])
valid_impact_chart = impact_df[~impact_df['AOV_Lift_%'].isna()]
ax5.bar(valid_impact_chart['Feature'], valid_impact_chart['AOV_Lift_%'], color='#2ecc71')
ax5.set_ylabel('AOV Lift (%)', fontsize=13, fontweight='bold')
ax5.set_xlabel('Feature', fontsize=13, fontweight='bold')
ax5.set_title('AOV Impact by Feature', fontsize=15, fontweight='bold')
ax5.grid(axis='y', alpha=0.3)
plt.setp(ax5.xaxis.get_majorticklabels(), rotation=45, ha='right')

for i, v in enumerate(valid_impact_chart['AOV_Lift_%']):
    ax5.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('GetYourGuide Pricing Features - Executive Summary', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
display(fig)
plt.close()

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)


In [0]:
# ============================================================================
# GETYOURGUIDE PRICING FEATURES AUDIT - COMPLETE REPORT
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from decimal import Decimal

plt.style.use('default')
sns.set_palette("husl")

print("="*80)
print("GETYOURGUIDE PRICING FEATURES AUDIT - COMPREHENSIVE REPORT")
print("="*80)

# ============================================================================
# 1. EXECUTIVE SUMMARY
# ============================================================================

summary_query = """
SELECT 
  'MARKETPLACE OVERVIEW' as section,
  'Total Active Tours' as metric,
  COUNT(DISTINCT tour_id) as value,
  NULL as pct_of_tours
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'MARKETPLACE OVERVIEW', 'Total Active Suppliers', COUNT(DISTINCT supplier_id), NULL
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'MARKETPLACE OVERVIEW', 'Total Tour Options', SUM(num_tour_options), NULL
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'INDIVIDUAL PRICING', 'Tours with Individual Pricing', SUM(CAST(has_individual_pricing AS INT)),
ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'INDIVIDUAL PRICING', 'Avg Individual Categories per Tour', ROUND(AVG(avg_num_individual_categories), 2), NULL
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'INDIVIDUAL PRICING', 'Tours with Scale Pricing on Individual', SUM(CAST(has_scale_pricing_individual AS INT)),
ROUND(100.0 * SUM(CAST(has_scale_pricing_individual AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'GROUP PRICING', 'Tours with Group Pricing', SUM(CAST(has_group_pricing AS INT)),
ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'GROUP PRICING', 'Tours with Scale Pricing on Group', SUM(CAST(has_scale_pricing_group AS INT)),
ROUND(100.0 * SUM(CAST(has_scale_pricing_group AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'ADDONS', 'Tours with Addons Configured', SUM(CAST(has_addons_configured AS INT)),
ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'SCALE PRICING', 'Tours with Any Scale Pricing', SUM(CAST(has_any_scale_pricing AS INT)),
ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'OPERATIONAL FEATURES', 'Tours with Cutoff Time Configured', SUM(CAST(has_cutoff_configured AS INT)),
ROUND(100.0 * SUM(CAST(has_cutoff_configured AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

UNION ALL SELECT 'OPERATIONAL FEATURES', 'Tours with Capacity Limits', SUM(CAST(has_capacity_limits AS INT)),
ROUND(100.0 * SUM(CAST(has_capacity_limits AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_summary

ORDER BY section, metric
"""

summary_df = spark.sql(summary_query).toPandas()

print("\n" + "="*80)
print("EXECUTIVE SUMMARY")
print("="*80)
display(summary_df)

# ============================================================================
# 2. INDIVIDUAL PRICING CATEGORY BREAKDOWN
# ============================================================================

print("\n" + "="*80)
print("INDIVIDUAL PRICING CATEGORIES - WHAT CATEGORIES DO SUPPLIERS USE?")
print("="*80)

ind_cat_df = spark.sql("""
SELECT 
  ticket_category,
  min_age,
  max_age,
  booking_type,
  num_price_tiers,
  num_tours_using,
  avg_tiers_per_category
FROM production.supply_analytics.individual_category_analysis
ORDER BY num_tours_using DESC
LIMIT 20
""").toPandas()

display(ind_cat_df)

# Chart: Top Individual Categories
fig, ax = plt.subplots(figsize=(14, 8))
top_cats = ind_cat_df.head(15)
ax.barh(range(len(top_cats)), top_cats['num_tours_using'], color='#3498db')
ax.set_yticks(range(len(top_cats)))
ax.set_yticklabels([f"{row['ticket_category']} ({row['min_age']}-{row['max_age']})" 
                     for _, row in top_cats.iterrows()])
ax.set_xlabel('Number of Tours Using', fontsize=13, fontweight='bold')
ax.set_title('Top 15 Individual Pricing Categories', fontsize=15, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 3. INDIVIDUAL PRICING TIER STRUCTURE
# ============================================================================

print("\n" + "="*80)
print("INDIVIDUAL PRICING TIERS - WHAT DO PRICE TIERS LOOK LIKE?")
print("="*80)

ind_tiers_df = spark.sql("""
SELECT 
  ticket_category,
  max_participants,
  num_tours,
  avg_retail_price,
  median_retail_price,
  min_retail_price,
  max_retail_price
FROM production.supply_analytics.individual_pricing_tier_patterns
WHERE num_tours >= 100
ORDER BY ticket_category, max_participants
""").toPandas()

display(ind_tiers_df)

# Chart: Price Distribution by Tier
fig, ax = plt.subplots(figsize=(14, 8))
for cat in ind_tiers_df['ticket_category'].unique()[:5]:
    cat_data = ind_tiers_df[ind_tiers_df['ticket_category'] == cat]
    ax.plot(cat_data['max_participants'], cat_data['median_retail_price'], 
            marker='o', linewidth=2.5, markersize=8, label=cat)

ax.set_xlabel('Max Participants (Tier Threshold)', fontsize=13, fontweight='bold')
ax.set_ylabel('Median Retail Price (EUR)', fontsize=13, fontweight='bold')
ax.set_title('Pricing Tier Patterns by Category', fontsize=15, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 4. GROUP PRICING ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("GROUP PRICING ANALYSIS")
print("="*80)

group_df = spark.sql("""
SELECT *
FROM production.supply_analytics.group_pricing_analysis
ORDER BY num_tours_using DESC
""").toPandas()

display(group_df)

group_tiers_df = spark.sql("""
SELECT 
  is_additional_pax,
  max_participants,
  num_tours,
  avg_retail_price,
  median_retail_price
FROM production.supply_analytics.group_pricing_tier_patterns
WHERE num_tours >= 50
ORDER BY is_additional_pax, max_participants
""").toPandas()

display(group_tiers_df)

# ============================================================================
# 5. ADDON ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("ADDON ANALYSIS - WHAT ADDONS ARE CONFIGURED?")
print("="*80)

addon_df = spark.sql("""
SELECT 
  addon_id,
  num_price_tiers,
  num_tours_using,
  avg_tiers_per_addon
FROM production.supply_analytics.addon_analysis
ORDER BY num_tours_using DESC
LIMIT 30
""").toPandas()

display(addon_df)

# Chart: Addon Adoption
fig, ax = plt.subplots(figsize=(12, 6))
top_addons = addon_df.head(20)
ax.bar(range(len(top_addons)), top_addons['num_tours_using'], color='#9b59b6')
ax.set_xticks(range(len(top_addons)))
ax.set_xticklabels(top_addons['addon_id'], rotation=45, ha='right')
ax.set_ylabel('Number of Tours', fontsize=13, fontweight='bold')
ax.set_xlabel('Addon ID', fontsize=13, fontweight='bold')
ax.set_title('Top 20 Addons by Usage', fontsize=15, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

addon_tiers_df = spark.sql("""
SELECT *
FROM production.supply_analytics.addon_pricing_tier_patterns
ORDER BY max_participants
""").toPandas()

display(addon_tiers_df)

# ============================================================================
# 6. CATEGORY COMBINATION PATTERNS
# ============================================================================

print("\n" + "="*80)
print("CATEGORY COMBINATION PATTERNS - WHAT COMBINATIONS ARE USED TOGETHER?")
print("="*80)

combo_df = spark.sql("""
SELECT 
  category_combination,
  num_categories_in_combo,
  num_tours_using
FROM production.supply_analytics.category_combination_patterns
ORDER BY num_tours_using DESC
LIMIT 30
""").toPandas()

display(combo_df)

# Chart: Distribution of Category Combinations
fig, ax = plt.subplots(figsize=(10, 6))
combo_counts = combo_df.groupby('num_categories_in_combo')['num_tours_using'].sum().reset_index()
ax.bar(combo_counts['num_categories_in_combo'], combo_counts['num_tours_using'], 
       color='#f39c12', edgecolor='black', linewidth=1.5)
ax.set_xlabel('Number of Categories in Combination', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Tours', fontsize=13, fontweight='bold')
ax.set_title('Distribution of Pricing Category Combinations', fontsize=15, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# 7. FEATURE ADOPTION HEATMAP BY LOCATION
# ============================================================================

print("\n" + "="*80)
print("FEATURE ADOPTION BY LOCATION")
print("="*80)

location_features_df = spark.sql("""
SELECT 
  location_name,
  COUNT(DISTINCT tour_id) as total_tours,
  ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1) as individual_pct,
  ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1) as group_pct,
  ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1) as addon_pct,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_pct,
  ROUND(100.0 * SUM(CAST(has_capacity_limits AS INT)) / COUNT(*), 1) as capacity_pct
FROM production.supply_analytics.tour_feature_summary
GROUP BY location_name
HAVING COUNT(DISTINCT tour_id) >= 100
ORDER BY total_tours DESC
LIMIT 20
""").toPandas()

display(location_features_df)

# Heatmap
fig, ax = plt.subplots(figsize=(14, 10))
heatmap_data = location_features_df[['individual_pct', 'group_pct', 'addon_pct', 
                                      'scale_pct', 'capacity_pct']].values.T

im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(location_features_df)))
ax.set_yticks(range(5))
ax.set_xticklabels(location_features_df['location_name'], rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(['Individual', 'Group', 'Addons', 'Scale', 'Capacity'], fontsize=12)

for i in range(5):
    for j in range(len(location_features_df)):
        ax.text(j, i, f'{heatmap_data[i, j]:.0f}%', ha="center", va="center", 
                color="black", fontsize=9, fontweight='bold')

ax.set_title('Feature Adoption by Top 20 Locations (%)', fontsize=15, fontweight='bold')
fig.colorbar(im, ax=ax, label='Adoption Rate (%)')
plt.tight_layout()
display(fig)
plt.close()

print("\n" + "="*80)
print("AUDIT COMPLETE - ALL FEATURE DETAILS ANALYZED")
print("="*80)
